In [35]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()

True

In [36]:
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [37]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [51]:
import os
import shutil

persist_directory = "chroma_db_demo"
collection_name = "sample"

if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory, ignore_errors=True)

vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001"),
    persist_directory=persist_directory,
    collection_name=collection_name,
)

#### Add documents in vector store.

In [52]:
inserted_ids = vector_store.add_documents(docs)
inserted_ids

['35d070e8-8362-416b-b8fc-11ab141971c7',
 '6f5b3d8b-05d3-4fcf-9f27-b16a573b44c9',
 '11c7cdde-46a1-4825-a280-7c97ae3d49fc',
 'f726d457-b11e-4c14-996d-5ba32cdee880',
 '0467809f-69c1-45cc-a013-261640c96ae6']

#### View Documents

In [53]:
vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['52a91de8-1203-496b-be33-997e15d0ead0',
  'c5b98da7-b079-4e24-b77d-e3d4bed6eb01',
  '23938909-bd2e-436f-b81c-4a95cffc9562',
  'c52b9c97-63a6-4932-bbca-01a7a5833ca6',
  '1451f9c3-42f5-4ba3-9d67-aaab9673b1f7',
  'b60cedad-7683-43e6-aee8-78b7f4ff2199',
  'b76baacc-d80e-43a1-b277-e220ba8e8490',
  'cd17b2f8-bf2f-4c31-819a-eb4c93822abf',
  '51cb15ce-89a6-41da-ae21-66be445e6fe7',
  '47b00c4a-a10c-4766-a105-98ddb2b146b2',
  '35d070e8-8362-416b-b8fc-11ab141971c7',
  '6f5b3d8b-05d3-4fcf-9f27-b16a573b44c9',
  '11c7cdde-46a1-4825-a280-7c97ae3d49fc',
  'f726d457-b11e-4c14-996d-5ba32cdee880',
  '0467809f-69c1-45cc-a013-261640c96ae6'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        ...,
        [-0.01358705, -0.0035

#### Search documents

In [42]:
vector_store.similarity_search(
    query='Who among these are a batsmen?',
    k=2
)

[Document(id='1451f9c3-42f5-4ba3-9d67-aaab9673b1f7', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
 Document(id='52a91de8-1203-496b-be33-997e15d0ead0', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.')]

#### Search with similarity score

In [43]:
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='c52b9c97-63a6-4932-bbca-01a7a5833ca6', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053),
 (Document(id='1451f9c3-42f5-4ba3-9d67-aaab9673b1f7', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.660536527633667)]

#### metadata filtering

In [45]:
vector_store.similarity_search_with_score(
    query="players from Chennai Super Kings",
    filter={'team': 'Chennai Super Kings'}
)

[(Document(id='1451f9c3-42f5-4ba3-9d67-aaab9673b1f7', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.5057892203330994),
 (Document(id='23938909-bd2e-436f-b81c-4a95cffc9562', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.5680614709854126)]

#### Update documents

In [ ]:
update_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)
vector_store.update_document(document_id=inserted_ids[0], document=update_doc1)

#### View documents

In [ ]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['3b6871f3-45e2-4525-8c1f-340e2e8db287',
  'a8034449-d4e9-44e2-ba03-8670e4d5426f',
  '0de2c17e-ace1-4933-98bf-0aa67b6d3eb1',
  'c704c5f9-8657-4d20-bec9-e1ed1d9178b5',
  '660fefc8-cbf0-4302-b84b-4fb104d418a4',
  '3ee3521c-58d7-4b5d-8e0d-3e37064b1dc5',
  '1fce22b4-11ab-44b4-96b9-c6f53d670198',
  'b7687732-e71a-4418-97d6-8cb494d005e8',
  'ea711ed4-6c5a-46a6-bd4b-4fe39ac4fc2d'],
 'embeddings': array([[ 0.02001663, -0.01274028, -0.02455907, ...,  0.00292782,
         -0.01331383,  0.00581167],
        [ 0.01581644,  0.00338993, -0.02235094, ...,  0.00195286,
         -0.07188983, -0.00888234],
        [ 0.01016247, -0.0516265 , -0.02895994, ...,  0.03298408,
         -0.02099222,  0.01782753],
        ...,
        [ 0.01581644,  0.00338993, -0.02235094, ...,  0.00195286,
         -0.07188983, -0.00888234],
        [ 0.01016247, -0.0516265 , -0.02895994, ...,  0.03298408,
         -0.02099222,  0.01782753],
        [-0.00089462,  0.01971646, -0.00964439, ...,  0.00234126,
         -0

#### Delete documents

In [ ]:
vector_store.delete(ids=[inserted_ids[1]])

In [ ]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['52a91de8-1203-496b-be33-997e15d0ead0',
  'c5b98da7-b079-4e24-b77d-e3d4bed6eb01',
  '23938909-bd2e-436f-b81c-4a95cffc9562',
  'c52b9c97-63a6-4932-bbca-01a7a5833ca6',
  '1451f9c3-42f5-4ba3-9d67-aaab9673b1f7'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m